# Parameter NetCDFs → Excel

**A. Lefauve, 2026**

Reads the per-case parameter NetCDFs (both the time-averaged `<CASE>.nc`
and the time-instant `<CASE>_<tStamp>.nc`) and writes:

- one Excel workbook per `.nc` (scalar parameters on the first sheet,
  arrays/spectra on later sheets);
- an aggregated `all_params.xlsx` with one row per case × file-type
  (time-averaged or instant) collecting every scalar parameter.

Works against either the Andes/Orion source structure
(`<ROOT>/<CASE>/001_Final/<CASE>.nc`) or a flat local folder
where every parameter `.nc` is in one directory.

In [1]:
# Imports
import re
import sys
from pathlib import Path

import h5py
import numpy as np
import pandas as pd

# Install openpyxl if missing
try:
    import openpyxl  # noqa: F401
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                          '--user', '--quiet', 'openpyxl'])
    import openpyxl  # noqa: F401

print('OK')

OK


## 1. Configure paths

Default: flat folder containing all `.nc` parameter files (one per case,
two per case if you have both time-averaged and time-instant). Each
`.nc` becomes one `.xlsx` in the output folder, plus an aggregated
`all_params.xlsx` collecting every scalar parameter across cases.

In [8]:
# Edit this one line to point at your folder of parameter NetCDFs.
NC_DIR = Path('/ccs/home/lefauve/git/INCITE/adrien/params')

# Where to write the .xlsx files (default: an 'xlsx' subfolder inside NC_DIR)
OUT = NC_DIR / 'xlsx'
OUT.mkdir(parents=True, exist_ok=True)

# Optional advanced mode: scan an Andes-style per-case tree
# (<root>/<CASE>/001_Final/<CASE>.nc) instead of a flat folder.
# Uncomment the next two lines and set ROOT to your tree root.
# ROOT = Path('/lustre/orion/cfd135/proj-shared/Hsst')
# NC_DIR = None   # signals 'use ROOT instead of NC_DIR'

print(f'NC_DIR = {NC_DIR}')
print(f'OUT    = {OUT}')
print(f'NC_DIR exists: {NC_DIR.exists() if NC_DIR else "(using ROOT)"}')

NC_DIR = /ccs/home/lefauve/git/INCITE/adrien/params
OUT    = /ccs/home/lefauve/git/INCITE/adrien/params/xlsx
NC_DIR exists: True


## 2. Discover parameter NetCDFs

Matches `<CASE>.nc` and `<CASE>_<tStamp>.nc`. Rejects slice files like
`R1P1_xy_z3960_st4x4_u.nc`.

In [9]:
CASES = ['R1P1','R1P7','R1P50','R4P1','R4P7','R4P50',
         'R6P1','R6P7','R6P50','R8P1','R8P7','R10P1','R10P7']

PARAM_RE = re.compile(r'^(?P<case>R\d+P\d+)(?:_(?P<tstamp>\d+(?:\.\d+)?))?\.nc$')

def discover(directory=None, root=None):
    """Yield (case, tstamp, path) for every parameter .nc found.

    Pass `directory=` for a flat folder, or `root=` for the Andes
    per-case tree (<root>/<CASE>/001_Final/).",
    """
    found = []
    if directory is not None:
        candidates = list(directory.iterdir())
    elif root is not None:
        candidates = []
        for case in CASES:
            d = root / case / '001_Final'
            if d.exists():
                candidates.extend(d.iterdir())
    else:
        raise ValueError('Pass either directory= or root=')
    for f in candidates:
        if not f.is_file():
            continue
        m = PARAM_RE.match(f.name)
        if m:
            found.append((m.group('case'), m.group('tstamp') or '', f))
    return sorted(found, key=lambda t: (CASES.index(t[0]) if t[0] in CASES else 99, t[1]))

# Use NC_DIR if set, otherwise ROOT
if NC_DIR is not None:
    files = discover(directory=NC_DIR)
else:
    files = discover(root=ROOT)

print(f'Found {len(files)} parameter NetCDFs:')
for case, tstamp, f in files:
    kind = 'instant' if tstamp else 'time-avg'
    print(f'  {case:<8} {kind:<8} {f.name:<32} {f.stat().st_size/1024:.1f} KB')

Found 26 parameter NetCDFs:
  R1P1     time-avg R1P1.nc                          24.7 KB
  R1P1     instant  R1P1_1100.000000.nc              76.1 KB
  R1P7     time-avg R1P7.nc                          24.7 KB
  R1P7     instant  R1P7_28.989491.nc                130.1 KB
  R1P50    time-avg R1P50.nc                         24.7 KB
  R1P50    instant  R1P50_56.563110.nc               303.4 KB
  R4P1     time-avg R4P1.nc                          24.7 KB
  R4P1     instant  R4P1_270.006042.nc               94.1 KB
  R4P7     time-avg R4P7.nc                          24.7 KB
  R4P7     instant  R4P7_161.636383.nc               202.1 KB
  R4P50    time-avg R4P50.nc                         24.7 KB
  R4P50    instant  R4P50_117.481239.nc              508.1 KB
  R6P1     time-avg R6P1.nc                          24.7 KB
  R6P1     instant  R6P1_96.533020.nc                130.1 KB
  R6P7     time-avg R6P7.nc                          24.7 KB
  R6P7     instant  R6P7_55.809399.nc               

## 3. Per-file Excel export

In [10]:
def nc_to_xlsx(nc_path, xlsx_path):
    """Read an HDF5/NetCDF parameter file and write a multi-sheet Excel.

    Sheet 'parameters': all scalar datasets (one row per parameter).
    Other sheets: non-scalar datasets (spectra, time series, etc.),
    one column per dataset.
    """
    with h5py.File(nc_path, 'r') as f:
        scalars, arrays = {}, {}
        def visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                data = obj[()]
                arr = np.asarray(data)
                if arr.size == 1:
                    scalars[name] = arr.item()
                else:
                    arrays[name] = arr.flatten()
        f.visititems(visit)
        # also pick up top-level datasets (visititems doesn't always cover them)
        for k in f.keys():
            if k in scalars or k in arrays:
                continue
            obj = f[k]
            if isinstance(obj, h5py.Dataset):
                arr = np.asarray(obj[()])
                if arr.size == 1:
                    scalars[k] = arr.item()
                else:
                    arrays[k] = arr.flatten()

    with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
        # Scalar parameters first
        if scalars:
            pd.DataFrame({'parameter': list(scalars), 'value': list(scalars.values())}) \
              .to_excel(writer, sheet_name='parameters', index=False)
        for name, arr in arrays.items():
            sheet = name.replace('/', '_')[:31]   # Excel sheet-name limit
            pd.DataFrame({name: arr}).to_excel(writer, sheet_name=sheet, index=False)
    return scalars, arrays

# Test on a single file:
if files:
    case, tstamp, src = files[0]
    dst = OUT / (src.stem + '.xlsx')
    s, a = nc_to_xlsx(src, dst)
    print(f'{src.name}  →  {dst}')
    print(f'  {len(s)} scalar params, {len(a)} array datasets')
    print(f'  scalars: {list(s)[:8]}{"..." if len(s) > 8 else ""}')

R1P1.nc  →  /ccs/home/lefauve/git/INCITE/adrien/params/xlsx/R1P1.xlsx
  26 scalar params, 0 array datasets
  scalars: ['1', 'D', 'Delta', 'Ek', 'Ep', 'Fr', 'Gamma1', 'Gamma2']...


## 4. Process all parameter NetCDFs

In [11]:
all_rows = []   # for the aggregated table

for case, tstamp, src in files:
    dst = OUT / (src.stem + '.xlsx')
    print(f'→ {src.name}')
    scalars, arrays = nc_to_xlsx(src, dst)
    row = {'case': case, 'kind': 'instant' if tstamp else 'time-avg', 'tStamp': tstamp}
    row.update(scalars)
    all_rows.append(row)

print(f'\nProcessed {len(all_rows)} files. Wrote per-file xlsx to {OUT}')

→ R1P1.nc
→ R1P1_1100.000000.nc
→ R1P7.nc
→ R1P7_28.989491.nc
→ R1P50.nc
→ R1P50_56.563110.nc
→ R4P1.nc
→ R4P1_270.006042.nc
→ R4P7.nc
→ R4P7_161.636383.nc
→ R4P50.nc
→ R4P50_117.481239.nc
→ R6P1.nc
→ R6P1_96.533020.nc
→ R6P7.nc
→ R6P7_55.809399.nc
→ R6P50.nc
→ R6P50_51.658600.nc
→ R8P1.nc
→ R8P1_95.780502.nc
→ R8P7.nc
→ R8P7_42.000092.nc
→ R10P1.nc
→ R10P1_252.420502.nc
→ R10P7.nc
→ R10P7_48.210110.nc

Processed 26 files. Wrote per-file xlsx to /ccs/home/lefauve/git/INCITE/adrien/params/xlsx


## 5. Aggregated `all_params.xlsx` — params as rows, cases as columns

Three sheets:

- **`time-avg`** — every parameter as a row, the 13 cases across columns
  (matches the layout you used in the old spreadsheet). Includes a sanity-
  check section at the bottom that recomputes physically-derived ratios.
- **`instant`** — same shape, for the time-instant snapshots.
- **`raw`** — the tidy long-format table (one row per file) for anyone
  who prefers that.

In [12]:
import numpy as np

agg = pd.DataFrame(all_rows)
front = ['case', 'kind', 'tStamp']
agg = agg[front + [c for c in agg.columns if c not in front]]

# ----- preferred parameter order (rows of the transposed view) -----
PREFERRED_ORDER = [
    'nu', 'D', 'Pr', 'g_rho0', 't',
    'Lk', 'Lb', 'Delta', 'kmax', 'kmaxLb',
    'Fr', 'Ri', 'Gn', 'Res', 'Re',
    'Ss', 'Ns', 'LambdaS', 'Gamma1', 'Gamma2',
    'Nx',
    'Ek', 'pb', 'ps', 'ek', 'Ep', 'ep',
]

def transpose_view(df_subset, kind_label):
    """Cases → columns, params → rows. Returns a DataFrame."""
    df = df_subset.set_index('case').drop(columns=['kind', 'tStamp'], errors='ignore')
    # Drop the spurious '1' dataset that some NCs have
    df = df.drop(columns=[c for c in df.columns if c == '1' or c == 1], errors='ignore')
    # Order columns by the case list
    df = df.reindex(CASES)
    # Order params (rows) by PREFERRED_ORDER, plus any extras at the end
    extras = [p for p in df.columns if p not in PREFERRED_ORDER]
    ordered = [p for p in PREFERRED_ORDER if p in df.columns] + sorted(extras)
    return df[ordered].T   # transpose so params are rows

def sanity_check_block(t):
    """Recompute physical-consistency ratios from the transposed time-avg view.
    Returns a small DataFrame to append below the parameters."""
    def safe(name): return pd.to_numeric(t.loc[name], errors='coerce') if name in t.index else None
    nu, D, Ri, Gn = safe('nu'), safe('D'), safe('Ri'), safe('Gn')
    Lk, Lb, Fr, Ek = safe('Lk'), safe('Lb'), safe('Fr'), safe('Ek')
    ek, ep, pb, G1, G2 = safe('ek'), safe('ep'), safe('pb'), safe('Gamma1'), safe('Gamma2')
    rows = {}
    if all(x is not None for x in (ek, nu, Ri, Gn)):
        rows['ek/(nu*Ri) / Gn = 1?']           = (ek / (nu * Ri)) / Gn
    if all(x is not None for x in (nu, ek, Lk)):
        rows['(nu^3/ek)^(1/4) / Lk = 1?']      = (nu**3 / ek)**0.25 / Lk
    if all(x is not None for x in (nu, D, ek, Lb)):
        rows['(nu*D^2/ek)^(1/4) / Lb = 1?']    = (nu * D**2 / ek)**0.25 / Lb
    if all(x is not None for x in (ek, Ri, Ek, Fr)):
        rows['ek/(sqrt(Ri)*Ek) / Fr = 1?']     = (ek / (np.sqrt(Ri) * Ek)) / Fr
    if all(x is not None for x in (ep, ek, G1)):
        rows['ep/ek / Gamma1 = 1?']            = (ep / ek) / G1
    if all(x is not None for x in (pb, ek, G2)):
        rows['pb/ek / Gamma2 = 1?']            = (pb / ek) / G2
    return pd.DataFrame(rows).T if rows else pd.DataFrame()

# Build the two transposed sheets
tavg = transpose_view(agg[agg['kind'] == 'time-avg'].copy(), 'time-avg')
tins = transpose_view(agg[agg['kind'] == 'instant'].copy(), 'instant')

# Sanity checks computed from time-avg
checks = sanity_check_block(tavg)

# Insert section-header rows by reindexing with empty strings.
def with_section_headers(t, checks):
    blank = pd.DataFrame([[None] * t.shape[1]],
                         index=['SANITY CHECKS'],
                         columns=t.columns)
    if checks.empty:
        return t
    return pd.concat([t, blank, checks])

tavg_full = with_section_headers(tavg, checks)
tins_full = tins   # no sanity checks for instant (some params are NaN)

agg_path = OUT / 'all_params.xlsx'
with pd.ExcelWriter(agg_path, engine='openpyxl') as writer:
    tavg_full.to_excel(writer, sheet_name='time-avg')
    tins_full.to_excel(writer, sheet_name='instant')
    agg.to_excel(writer, sheet_name='raw', index=False)

print(f'Wrote {agg_path}')
print(f'  time-avg sheet: {tavg_full.shape[0]} rows × {tavg_full.shape[1]} cols')
print(f'  instant  sheet: {tins_full.shape[0]} rows × {tins_full.shape[1]} cols')
print(f'  raw      sheet: {agg.shape[0]} rows × {agg.shape[1]} cols')
tavg_full

Wrote /ccs/home/lefauve/git/INCITE/adrien/params/xlsx/all_params.xlsx
  time-avg sheet: 33 rows × 13 cols
  instant  sheet: 26 rows × 13 cols
  raw      sheet: 26 rows × 30 cols


/tmp/ipykernel_626/462837547.py:65: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([t, blank, checks])


case,R1P1,R1P7,R1P50,R4P1,R4P7,R4P50,R6P1,R6P7,R6P50,R8P1,R8P7,R10P1,R10P7
nu,0.000498,0.000498,0.000498,0.000250,0.000250,0.000250,0.000125,0.000125,0.000125,0.000031,0.000031,0.000020,0.000020
D,0.000498,0.000071,0.000010,0.000250,0.000036,0.000005,0.000125,0.000018,0.000003,0.000031,0.000004,0.000020,0.000003
Pr,1.000000,7.000000,50.000000,1.000000,7.000000,50.000000,1.000000,7.000000,50.000000,1.000000,7.000000,1.000000,7.000000
g_rho0,163.076041,169.298191,165.103185,153.035721,153.003631,152.446185,123.342519,127.360435,121.047964,150.788656,156.073715,129.631047,153.464256
t,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Lk,0.014469,0.014415,0.014323,0.008422,0.008375,0.008360,0.004941,0.004968,0.004938,0.001893,0.001868,0.001312,0.001334
Lb,0.014469,0.005448,0.002026,0.008422,0.003166,0.001182,0.004941,0.001878,0.000698,0.001893,0.000706,0.001312,0.000504
Delta,0.016362,0.002045,0.003142,0.012272,0.004909,0.001571,0.008181,0.002045,0.001047,0.002877,0.001091,0.002045,0.000793
kmax,1.130839,0.375392,1.551012,1.457069,1.550654,1.328614,1.655712,1.089354,1.499412,1.519874,1.545189,1.559351,1.573436
kmaxLb,2.778108,8.368842,2.025512,2.156105,2.025979,2.364564,1.897427,2.883905,2.095216,2.067009,2.033145,2.014680,1.996644
